# Bulk Sample Loading

Notebook to bulk load samples belonging to a single panel into the database.

Naming convention (sample number is always zero-padded to 2 digits):
- Panel `A_<number>` (e.g. `A_2`) -> samples `A-0<number>-<sample_number>` (e.g. `A-002-01`, `A-002-02`, ...)
- Panel `X-<number>` (e.g. `X-01`) -> samples `<panelname>-<sample_number>` (e.g. `X-01-01`, `X-01-02`, ...)

All samples share the same dimensions, so height/width/thickness are entered once. All samples are loaded with `keyhole = True` and `parallel_faces = True`.

## Imports

In [ ]:
import re
import psycopg2
from tabulate import tabulate
#import a folder in the parent directory
import sys
sys.path.append('../')
import dbtools.dbtools as qrs
import dbtools.load as load

## Connection

In [ ]:
try:
    # Connect to the PostgreSQL database
    conn = qrs.connect()
    print("Connected to the database")

except (Exception, psycopg2.DatabaseError) as error:
    print(error)

## Panel selection

In [ ]:
parent_table_name = 'panels'

parent_data = qrs.get_data(parent_table_name)

parent_data

In [ ]:
# Set the panel name, e.g. 'A_2' or 'X-01' (both '_' and '-' separators are supported)
panel_name = 'A_38'

match = re.match(r'^([A-Za-z]+)([_-])(\d+)$', panel_name)
assert match, f"Panel name '{panel_name}' does not follow the '<letters>_<number>' or '<letters>-<number>' convention"

panel_prefix, panel_separator, panel_number = match.group(1), match.group(2), match.group(3)

# '_'-separated panels (e.g. 'A_2') get their number zero-padded to 3 digits for the sample name (e.g. 'A-002').
# '-'-separated panels (e.g. 'X-01') already carry their final number format, so it's reused as-is (e.g. 'X-01').
if panel_separator == '_':
    sample_prefix = f'{panel_prefix}-{panel_number.zfill(3)}'
else:
    sample_prefix = f'{panel_prefix}-{panel_number}'

panel_matches = parent_data[parent_data['name_panel'] == panel_name]
assert len(panel_matches) > 0, f"Panel '{panel_name}' not found in the database"
assert len(panel_matches) == 1, f"Multiple panels named '{panel_name}' found in the database"

panel_id = int(panel_matches['id_panel'].values[0])

print(f"Panel '{panel_name}' found with ID: {panel_id}")

## Sample data

Compulsory information:
- num_samples: Number of samples to create for this panel
- height: In mm
- width: In mm
- thickness: In mm

In [ ]:
num_samples = 17

height = 150
width = 40
thickness = 4.7

keyhole = True

parallel_faces = True

sample_names = [f'{sample_prefix}-{i:02d}' for i in range(1, num_samples + 1)]

sample_names

In [ ]:
for name in sample_names:

    load.load_sample(conn, name, panel_id, height, width, thickness, keyhole, parallel_faces)

## Make sure insert was correct

In [ ]:
data = qrs.get_data_metadata('samples')

data[data['panel_id_sample'] == panel_id]

In [ ]:
conn.close()